<a href="https://colab.research.google.com/github/MLuc123/ds2002-fa26/blob/main/2026_10_02_normalize_and_join_nested_json_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
orders_json =[
    {'order': 1, 'vendor_id': 'V-01', 'lines': [{'item': 'Cheeseburger', 'qty': 2}, {'item': 'Soda', 'qty': 1}]},
    {'order': 2, 'vendor_id': 'V-10', 'lines': [{'item': 'Foam Finger', 'qty': 1}]},
    {'order': 3, 'vendor_id': 'V-07', 'lines': [{'item': 'Hot Dog', 'qty': 3}]}
]
vendors_json = [
    {'vendor_id': 'V-01', 'name': 'Hoos Burgers', 'zone': 'A'},
    {'vendor_id': 'V-10', 'name': 'Cav Merch', 'zone': 'A'}
]

lines=pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
lines

assert len(lines) == 4
assert lines['qty'].sum() == 7
print('lines')

lines


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
# TODO
vendors_df=pd.DataFrame(vendors_json)
joined=lines.merge(vendors_df, on='vendor_id', how='left', indicator=True)
unmatched=joined[joined['_merge']=='left_only']
unmatched_qty=unmatched['qty'].sum()

print(joined)
print(f"\nVendor{unmatched['vendor_id'].iloc[0]} did not match.")
print(f"there are {unmatched_qty} units attached to it.")

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

VendorV-07 did not match.
there are 3 units attached to it.


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
# TODO
joined['name']=joined['name'].fillna('Unknown')
qty_per_vendor=joined.groupby('name')['qty'].sum()
print(qty_per_vendor)

# I chose to fill the missing vendor names with unknown vendor before goruping. If I didn't do this the NaN name would be dropped by default we would lose track of those three items in teh total count.

name
Cav Merch       1
Hoos Burgers    3
Unknown         3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
# TODO
qty_per_zone = joined.groupby('zone')['qty'].sum()
print (qty_per_zone)

print(f'\nTotal items assigned to a zone:{qty_per_zone.sum()}')
print("the difference of three went to vendor V-07 which didnt have a match in the vendor's table and therefore has a NaN assignment")

zone
A    4
Name: qty, dtype: int64

Total items assigned to a zone:4
the difference of three went to vendor V-07 which didnt have a match in the vendor's table and therefore has a NaN assignment


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
import pandas as pd
import numpy as np

ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},
]

# 1. Pre-process the JSON to give Order 5 an empty line
for order in ragged_json:
    if 'lines' not in order:
        order['lines'] = [{'item': np.nan, 'qty': np.nan}]

# 2. Now normalize (errors='ignore' is no longer needed)
ragged_df = pd.json_normalize(ragged_json, record_path='lines', meta=['order', 'vendor_id'])

print(ragged_df)


    item  qty order vendor_id
0   Soda  2.0     4      V-01
1    NaN  NaN     5      V-10
2  Fries  NaN     6      V-01


### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

Having one row per line makes it easy to group data. If we kept it as one row per oder the items and quantities would be stuck inside lists and dictionaries. This would make it hard to answer simple questions because many loops would be needed to go through ever list inside each row order.

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A quantity of zero means that the customer explicityl didn't want the item, or might have canceled a specific part of the order. A missing quantity means we don't know how many they ordered - maybe its a system error. If you use something like fillna(0) the data will be permanently altered to say they ordered zero fries, this throws off the data because they might have actually ordered 1 of 2.

_your answer here_